In [62]:
import pandas as pd

pd.set_option("display.max_rows", None)

goods_no = "5255062"
df = pd.read_csv(f"musinsa_reviews_{goods_no}.csv", encoding="utf-8-sig")

print(df[['review', 'star']])

                                                 review  star
0                이너로 입으려 구매했는데 원단퀄리티도 좋고 세탁후 변형없어요\n좋아요   5.0
1                      그냥 재질 괜찮고 핏도 그냥 막 옷이 길지도 않고 괜찮은듯   5.0
2                          가성비 좋은듯 그냥 재질도괜찮고 걍 기본티 이거 ㄱ   5.0
3           딱 근본이네요 좋습니다 가격도 무난하고 여름에 더 많은 수량을 구매할거 같아요   5.0
4             중학생 아들 체육복 이너용으로 구매했습니다. 가성비좋고 막입히기 좋습니다.   5.0
5                         이어티로 너무 잘 입고 있어요 다음에 또 사고 싶어요   5.0
6                         기본티로 좋아여 이너로 잘입고다녀요 체형에 잘 맞아요   5.0
7                       따구이너로 입기 적당하고 좋습니다 원단이 은근 쫀쫀하내요   4.0
8                 한번 세탁 돌리니까 목이 좀 쭈글쭈글해지네요\n그거 빼곤 만족합니다   4.0
9                           남방 안에 입기 너무 좋아요@ 두께감도 적당합니다   5.0
10                      넉넉해서 셔츠안에 입기에도 좋고 단독으로 입기에도 좋아요   5.0
11                        소재도 디자인도 좋고 너무편합니다. 자주입을것같아요.   5.0
12        동생 이너로 입으라고 사줬는데 마음에 든다고 하네요 믿고 입는 무탠다드 반팔입니다   5.0
13                         줄어 드는 재질이예요 넉넉히 사이즈 감안하고 사세요   3.0
14    배송은 언제나 빠름~!! 몇번째 재구매인지 기억도 안남 ㅎ 면도 너무좋구 목카라도 ...   5.0
15      

In [63]:
df['star'].value_counts()

star
5.0    4383
4.0     472
3.0      67
1.0      15
2.0      13
Name: count, dtype: int64

In [64]:
df.isna().sum()

nickname         0
review           0
star            52
product_name     0
price            0
discount_pct     0
review_count     0
view_count       0
sales_count      0
dtype: int64

In [65]:
df['star'].mean()

np.float64(4.857575757575757)

In [66]:
df['star'] = df['star'].fillna(df['star'].mean())
df.isna().sum()

nickname        0
review          0
star            0
product_name    0
price           0
discount_pct    0
review_count    0
view_count      0
sales_count     0
dtype: int64

In [67]:
df['review'] = df['review'].map(lambda x: x.replace("\n", " "))
df['review']

0                   이너로 입으려 구매했는데 원단퀄리티도 좋고 세탁후 변형없어요 좋아요
1                        그냥 재질 괜찮고 핏도 그냥 막 옷이 길지도 않고 괜찮은듯
2                            가성비 좋은듯 그냥 재질도괜찮고 걍 기본티 이거 ㄱ
3             딱 근본이네요 좋습니다 가격도 무난하고 여름에 더 많은 수량을 구매할거 같아요
4               중학생 아들 체육복 이너용으로 구매했습니다. 가성비좋고 막입히기 좋습니다.
5                           이어티로 너무 잘 입고 있어요 다음에 또 사고 싶어요
6                           기본티로 좋아여 이너로 잘입고다녀요 체형에 잘 맞아요
7                         따구이너로 입기 적당하고 좋습니다 원단이 은근 쫀쫀하내요
8                    한번 세탁 돌리니까 목이 좀 쭈글쭈글해지네요 그거 빼곤 만족합니다
9                             남방 안에 입기 너무 좋아요@ 두께감도 적당합니다
10                        넉넉해서 셔츠안에 입기에도 좋고 단독으로 입기에도 좋아요
11                          소재도 디자인도 좋고 너무편합니다. 자주입을것같아요.
12          동생 이너로 입으라고 사줬는데 마음에 든다고 하네요 믿고 입는 무탠다드 반팔입니다
13                           줄어 드는 재질이예요 넉넉히 사이즈 감안하고 사세요
14      배송은 언제나 빠름~!! 몇번째 재구매인지 기억도 안남 ㅎ 면도 너무좋구 목카라도 ...
15                             면 이너티로 딱 1년정도 입기 괜찮은 품질같아요
16                          정석적인 흰티에다가 가격도 착하고 무난무난하게 좋았다
17            

In [68]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = "./kcbert_mlm/finetune"  # ← 여기가 최종 파인튜닝 결과

tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

In [69]:

import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model.to(device)
model.eval()

# 토크나이저 최대 길이 제한
tokenizer.model_max_length = 128

# 데이터 준비
texts = df["review"].astype(str).tolist()

# 배치 사이즈 조정 (메모리 부족하면 더 줄여라)
BATCH_SIZE = 16

# DataLoader 대신 간단 루프
all_preds = []

with torch.no_grad():
    for i in tqdm(range(0, len(texts), BATCH_SIZE)):
        batch_texts = texts[i:i + BATCH_SIZE]
        encoding = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        logits = model(**encoding).logits
        preds = logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().tolist())

        del encoding, logits, preds
        if device == "cuda":
            torch.cuda.empty_cache()

df["label"] = all_preds
print("완료:", len(df), "개의 리뷰에 라벨 부여됨")


Using device: cuda


100%|██████████| 313/313 [00:04<00:00, 66.25it/s]

완료: 5002 개의 리뷰에 라벨 부여됨


In [70]:
pd.set_option("display.max_rows", None)

In [71]:
df['label'].value_counts()

label
1    4816
0     186
Name: count, dtype: int64

In [72]:
pd.set_option("display.max_columns", None)
df = df[['review', 'star', 'label']]

In [73]:
# csv로 저장
df.to_csv(f"musinsa_reviews_label_{goods_no}_rebert.csv", index=False, encoding="utf-8-sig")